In [5]:
import os
import re
import math
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.dates import DateFormatter
from dateutil.tz import tzutc, tzlocal
from scipy import stats
from scipy.optimize import brentq, curve_fit, fsolve


import sys
import signal
import tempfile
import time
from collections import Counter
from IPython.display import clear_output

In [6]:
%load_ext autoreload
%autoreload 2

import official_GSSHA_Python_functions as gf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Load the paths for GSSHA model folder and executables

In [7]:
cwd = os.getcwd()
script_dir = Path.cwd() 

GSSHA_prj_name = "Waialua_FIM_testing"
model_dir = script_dir / GSSHA_prj_name
GSSHA_executables = script_dir / 'GSSHA_applications'

# Force shutdown of any GSSHA model running in model folder

In [8]:
 # Use  if you're in Jupyter or interactive session
process = gf.GSSHA_auto_shutdown(model_dir)

# When you need to force shutdown:
gf.force_shutdown_gssha(
    process=process,
    model_dir=model_dir
)

GSSHA could not be launched: [WinError 193] %1 is not a valid Win32 application
GSSHA stopped and gssha.exe is unlocked.


True

# Run flow testing for model development

In [ ]:
test_input_flow_results_dir = model_dir / "TEST_INPUT_FLOW_SIMULATIONS"


xys_tsf_iteration_file_name = GSSHA_prj_name + "_ITERATION"
xys_tsf_iteration_value = "155.0"



FLOW_TEST = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200]
tot_time = 1000


gf.cleanup_model_dir(model_dir)

for flow in FLOW_TEST:
    new_value = float(flow)
    cfs_flow_for_filenames = round(flow*35.31467)
    
    output_file = (
        test_input_flow_results_dir
        / f"{GSSHA_prj_name}_OUTPUT-input-flow-cfs-{cfs_flow_for_filenames}.tsf"
    )
    
    if output_file.exists():
        print(f"{output_file.name} already exists. Skipping...")
        continue

    
    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
    

    #ITERATION FILES
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + ".tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value =xys_tsf_iteration_value, new_value = new_value)
    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = xys_tsf_iteration_value, new_value = new_value)
    ##UPDATE THE ITERATION W THE NEW XYS AND TSF FILE

    #!!!!write a function to copy with specified cfs flows in name!!!

    #SAVE TEXT XYS file for knowing the input flows for model run
    #These files are for the 

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + ".tsf"
    )


    

    df_prj.loc['TOT_TIME', 'Value'] = tot_time
    
    df_prj = df_prj.reset_index()


    #UPDATE PRJ FILE WHICH CONTAINS THE PATHS
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )


    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    
    #this function allows you to see when model convergence occurs so you can quit the run
    return_code = gf.run_gssha_convergence_view(
    MODEL_DIR=model_dir,
    PROJECT_FILE=GSSHA_prj_name + ".prj",
    DEP_FILE=GSSHA_prj_name + ".dep",
    cell_size=10,
    dep_check_seconds=15,
    display_last_n=10
)
    # move_and_rename_gssha_output(MODEL_DIR, RESULTS_DIR, output_description = "TEST_RUN" + first_date, extension = "otl")
    gf.cleanup_model_dir(model_dir)

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".dep",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-timeseries-depth-m-" + str(cfs_flow_for_filenames) + ".dep"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".gfl",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT-maxflood-dep-m-" + str(cfs_flow_for_filenames) + ".gfl"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".oqc",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT_USGS-STATS-location-cms-" + str(cfs_flow_for_filenames) + ".oqc"
    )
    
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".ows",
        output_folder=test_input_flow_results_dir,
        new_filename=GSSHA_prj_name + "_OUTPUT_WSE-active-USGS-gauge-m-" + str(cfs_flow_for_filenames) + ".ows"
    )


GSSHA is running — DEP check 4
DEP file: Waialua_FIM_testing.dep
Use the GSSHA control window to stop the simulation.
Skipping incomplete DEP timestep 120.5: found 153,348 values; expected 153,438.

DEP timestep blocks read: 5


,timestep,cumulative_change,max_positive_change_per_cell,inundated_area_change
0,30.5,741.551026,3.303278,124700.0
1,60.5,526.332052,1.754419,33900.0
2,90.5,138.307617,0.582336,3100.0


Read Results from calibration/senstivity and run model for final stage height maps (datum offset with model parameters: input flow, simulation time)

In [ ]:
test_flow_results = pd.read_csv(test_input_flow_results_dir / "